# EXACT 2026 — Root Data Audit Notebook

**Goal:** audit `raw411` vs `clean/v5` from the root, without treating Z3 as an absolute judge.

This notebook focuses on the triangle:

```text
label  ↔  explanation  ↔  premises / FOL / idx
```

It prints all important summaries directly in the notebook and also saves CSV/JSON artifacts to `/kaggle/working`.

**Important principle:** Z3, if added later, should be a consistency checker / verifier, not a truth oracle for noisy data.

In [ ]:

# ==================================================================
# STAGE 0 -- Imports & path resolver
# ==================================================================
import os, re, json, csv, math, hashlib, textwrap, statistics
from pathlib import Path
from collections import Counter, defaultdict
from difflib import SequenceMatcher

try:
    import pandas as pd
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pandas'], check=False)
    import pandas as pd

pd.set_option('display.max_colwidth', 180)
pd.set_option('display.max_rows', 200)

print('='*80)
print('ROOT AUDIT NOTEBOOK -- imports OK')
print('='*80)

def _resolve(cands, label='path'):
    import glob
    expanded=[]
    for p in cands:
        hits = sorted(glob.glob(p, recursive=True)) if any(ch in p for ch in '*?[') else [p]
        expanded.extend(hits)
    for p in expanded:
        if os.path.exists(p):
            print(f'  resolved {label}: {p}')
            return p
    print(f'  WARNING {label}: none found; using first candidate: {cands[0]}')
    return cands[0]

RAW_PATH = _resolve([
    '/kaggle/input/**/Logic_Based_Educational_Queries.json',
    '/kaggle/input/**/Logic_Based_Educational_Queries(2).json',
    '/mnt/data/Logic_Based_Educational_Queries(2).json',
    '/mnt/data/Logic_Based_Educational_Queries.json',
], 'RAW411')

CLEAN_PATH = _resolve([
    '/kaggle/input/**/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json',
    '/kaggle/input/**/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy(1).json',
    '/mnt/data/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy(1).json',
    '/mnt/data/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json',
], 'CLEAN/V5')

OUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/mnt/data')
print('OUT_DIR:', OUT_DIR)

In [ ]:

# ==================================================================
# STAGE 1 -- Load data and helper functions
# ==================================================================
def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

raw = load_json(RAW_PATH)
clean = load_json(CLEAN_PATH)

print(f'RAW   records: {len(raw)}')
print(f'CLEAN records: {len(clean)}')

LABELS = ['A','B','C','D','YES','NO','UNKNOWN']

def norm_ws(s):
    return re.sub(r'\s+', ' ', str(s).strip())

def norm_text(s):
    s = norm_ws(s).lower()
    s = re.sub(r'[“”"\'`]', '', s)
    s = re.sub(r'\s+', ' ', s)
    return s

def norm_answer(a):
    x = str(a).strip()
    xu = x.upper()
    if xu in ('YES','TRUE'): return 'YES'
    if xu in ('NO','FALSE'): return 'NO'
    if xu in ('UNKNOWN','UNCERTAIN','CANNOT BE DETERMINED','UNDETERMINED'): return 'UNKNOWN'
    if xu in ('A','B','C','D'): return xu
    return xu

def h(s, n=16):
    return hashlib.md5(str(s).encode('utf-8')).hexdigest()[:n]

def rec_key_by_premises_questions(rec):
    prem = '||'.join(norm_text(x) for x in rec.get('premises-NL', []))
    qs = '||'.join(norm_text(x) for x in rec.get('questions', []))
    return h(prem + '###' + qs)

def q_key(rec, q_idx):
    prem = '||'.join(norm_text(x) for x in rec.get('premises-NL', []))
    q = norm_text(rec.get('questions', [''])[q_idx])
    return h(prem + '###' + q)

def flatten(dataset, name):
    rows=[]
    for ri, rec in enumerate(dataset):
        qs = rec.get('questions', []) or []
        ans = rec.get('answers', []) or []
        exps = rec.get('explanation', []) or []
        idxs = rec.get('idx', []) or []
        for qi, q in enumerate(qs):
            rows.append({
                'dataset': name,
                'record_i': ri,
                'q_idx': qi,
                'record_key': rec_key_by_premises_questions(rec),
                'q_key': q_key(rec, qi),
                'n_premises': len(rec.get('premises-NL', [])),
                'question': q,
                'answer': ans[qi] if qi < len(ans) else None,
                'answer_norm': norm_answer(ans[qi]) if qi < len(ans) else None,
                'explanation': exps[qi] if qi < len(exps) else '',
                'idx': idxs[qi] if qi < len(idxs) else None,
                'premises_NL': rec.get('premises-NL', []),
                'premises_FOL': rec.get('premises-FOL', []),
            })
    return pd.DataFrame(rows)

raw_q = flatten(raw, 'raw')
clean_q = flatten(clean, 'clean')

print(f'RAW questions   : {len(raw_q)}')
print(f'CLEAN questions : {len(clean_q)}')
print('
RAW label distribution:')
print(raw_q['answer_norm'].value_counts().reindex(LABELS, fill_value=0))
print('
CLEAN label distribution:')
print(clean_q['answer_norm'].value_counts().reindex(LABELS, fill_value=0))

In [ ]:

# ==================================================================
# STAGE 2 -- Distribution comparison
# ==================================================================
def dist_table(raw_q, clean_q):
    labels = sorted(set(raw_q.answer_norm.dropna()) | set(clean_q.answer_norm.dropna()), key=lambda x: LABELS.index(x) if x in LABELS else 999)
    rows=[]
    for lab in labels:
        r = int((raw_q.answer_norm == lab).sum())
        c = int((clean_q.answer_norm == lab).sum())
        rows.append({'label': lab, 'raw_count': r, 'clean_count': c, 'delta_clean_minus_raw': c-r})
    return pd.DataFrame(rows)

dist = dist_table(raw_q, clean_q)
print('='*80)
print('LABEL DISTRIBUTION SHIFT')
print('='*80)
display(dist)

print('
Question count delta:', len(clean_q) - len(raw_q))
print('Record count delta  :', len(clean) - len(raw))

summary = {
    'raw_records': len(raw),
    'clean_records': len(clean),
    'raw_questions': len(raw_q),
    'clean_questions': len(clean_q),
    'raw_label_counts': raw_q['answer_norm'].value_counts().to_dict(),
    'clean_label_counts': clean_q['answer_norm'].value_counts().to_dict(),
}

In [ ]:

# ==================================================================
# STAGE 3 -- Match raw vs clean by premise+question key and inspect answer changes
# ==================================================================
raw_small = raw_q[['q_key','record_i','q_idx','answer_norm','answer','question','explanation','idx','record_key']].rename(columns={
    'record_i':'raw_record_i','q_idx':'raw_q_idx','answer_norm':'raw_answer_norm','answer':'raw_answer',
    'question':'raw_question','explanation':'raw_explanation','idx':'raw_idx','record_key':'raw_record_key'
})
clean_small = clean_q[['q_key','record_i','q_idx','answer_norm','answer','question','explanation','idx','record_key']].rename(columns={
    'record_i':'clean_record_i','q_idx':'clean_q_idx','answer_norm':'clean_answer_norm','answer':'clean_answer',
    'question':'clean_question','explanation':'clean_explanation','idx':'clean_idx','record_key':'clean_record_key'
})

matched = raw_small.merge(clean_small, on='q_key', how='outer', indicator=True)
matched['answer_changed'] = (matched['_merge'].eq('both')) & (matched['raw_answer_norm'] != matched['clean_answer_norm'])

print('='*80)
print('RAW ↔ CLEAN QUESTION MATCHING')
print('='*80)
print(matched['_merge'].value_counts())
print('
Matched answer changes:', int(matched['answer_changed'].sum()))

trans = matched[matched['answer_changed']].groupby(['raw_answer_norm','clean_answer_norm']).size().reset_index(name='count').sort_values('count', ascending=False)
print('
Answer transition table:')
display(trans)

changes = matched[matched['answer_changed']].copy()
changes['raw_exp_short'] = changes['raw_explanation'].fillna('').map(lambda s: norm_ws(s)[:240])
changes['clean_exp_short'] = changes['clean_explanation'].fillna('').map(lambda s: norm_ws(s)[:240])
changes['q_short'] = changes['raw_question'].fillna(changes['clean_question']).map(lambda s: norm_ws(s)[:160])

print('
First 30 answer changes:')
display(changes[['raw_record_i','raw_q_idx','clean_record_i','clean_q_idx','raw_answer_norm','clean_answer_norm','q_short','raw_exp_short','clean_exp_short']].head(30))

changes.to_csv(OUT_DIR/'audit_label_changes_raw_vs_clean.csv', index=False)
print('Saved:', OUT_DIR/'audit_label_changes_raw_vs_clean.csv')

In [ ]:

# ==================================================================
# STAGE 4 -- Explanation-vs-label heuristic flags
# These are NOT automatic fixes. They are review candidates.
# ==================================================================
def infer_mc_from_explanation(exp):
    e = norm_ws(exp)
    patterns = [
        r'support(?:ing|s)?\s+option\s+([ABCD])',
        r'option\s+([ABCD])\s+(?:is|as)\s+(?:the\s+)?(?:correct|valid|logically valid|strongest|best)',
        r'making\s+([ABCD])\s+(?:the\s+)?(?:correct|valid|strongest|best)',
        r'hence,?\s+option\s+([ABCD])',
        r'therefore,?\s+option\s+([ABCD])',
        r'answer\s+is\s+([ABCD])\b',
    ]
    for pat in patterns:
        m = re.search(pat, e, flags=re.I)
        if m:
            return m.group(1).upper(), 'high', f'pattern:{pat}'
    return None, None, None

POS_PHRASES = [
    'does meet all requirements', 'meets all requirements', 'satisfying all conditions',
    'therefore, yes', 'so yes', 'the statement is true', 'is true', 'entails', 'follows',
    'qualifies', 'eligible', 'can ', 'allows him', 'allows her', 'confirms that'
]
NEG_PHRASES = [
    'does not meet all requirements', "doesn't meet all requirements", 'does not meet',
    'cannot ', 'can not ', 'is not eligible', 'not eligible', 'does not qualify', 'not qualify',
    'prevents', 'fails', 'insufficient', 'below the', 'not confirmed', 'no premise confirms'
]
UNK_PHRASES = [
    'uncertain', 'unknown', 'cannot be determined', 'not enough information', 'insufficient information',
    'not specified', 'not stated', 'cannot conclude', 'cannot infer', 'not enough evidence'
]

def infer_ynu_from_explanation(exp):
    e = norm_text(exp)
    if any(p in e for p in UNK_PHRASES):
        return 'UNKNOWN', 'medium', 'unknown_phrase'
    if any(p in e for p in NEG_PHRASES):
        return 'NO', 'medium', 'negative_phrase'
    if any(p in e for p in POS_PHRASES):
        return 'YES', 'low', 'positive_phrase'
    return None, None, None

def flag_explanation_disagreement(df, name):
    rows=[]
    for _, r in df.iterrows():
        ans = r['answer_norm']
        exp = r['explanation'] or ''
        q = r['question'] or ''
        hint = conf = reason = None
        if ans in list('ABCD'):
            hint, conf, reason = infer_mc_from_explanation(exp)
        elif ans in ('YES','NO','UNKNOWN'):
            hint, conf, reason = infer_ynu_from_explanation(exp)
        flag = bool(hint and hint != ans)
        rows.append({
            'dataset': name,
            'record_i': r['record_i'],
            'q_idx': r['q_idx'],
            'answer': ans,
            'explanation_hint': hint,
            'hint_confidence': conf,
            'hint_reason': reason,
            'flag_disagree': flag,
            'question': norm_ws(q)[:300],
            'explanation': norm_ws(exp)[:600],
        })
    return pd.DataFrame(rows)

raw_flags = flag_explanation_disagreement(raw_q, 'raw')
clean_flags = flag_explanation_disagreement(clean_q, 'clean')

print('='*80)
print('EXPLANATION ↔ LABEL HEURISTIC FLAGS')
print('='*80)
print('Raw flags   :', int(raw_flags.flag_disagree.sum()), '/', len(raw_flags))
print('Clean flags :', int(clean_flags.flag_disagree.sum()), '/', len(clean_flags))

print('
Top raw flags:')
display(raw_flags[raw_flags.flag_disagree].head(30))
print('
Top clean flags:')
display(clean_flags[clean_flags.flag_disagree].head(30))

raw_flags.to_csv(OUT_DIR/'audit_raw_explanation_label_flags.csv', index=False)
clean_flags.to_csv(OUT_DIR/'audit_clean_explanation_label_flags.csv', index=False)
print('Saved:', OUT_DIR/'audit_raw_explanation_label_flags.csv')
print('Saved:', OUT_DIR/'audit_clean_explanation_label_flags.csv')

In [ ]:

# ==================================================================
# STAGE 5 -- Index validity, FOL format, and record-level structural issues
# ==================================================================
def check_idx(rec, ri, dataset):
    n = len(rec.get('premises-NL', []))
    issues=[]
    for qi, idx in enumerate(rec.get('idx', []) or []):
        if idx is None:
            issues.append({'dataset':dataset,'record_i':ri,'q_idx':qi,'issue':'idx_missing','idx':idx,'n_premises':n})
            continue
        if not isinstance(idx, list):
            issues.append({'dataset':dataset,'record_i':ri,'q_idx':qi,'issue':'idx_not_list','idx':idx,'n_premises':n})
            continue
        if len(idx)==0:
            issues.append({'dataset':dataset,'record_i':ri,'q_idx':qi,'issue':'idx_empty','idx':idx,'n_premises':n})
        for k in idx:
            if not isinstance(k, int):
                issues.append({'dataset':dataset,'record_i':ri,'q_idx':qi,'issue':'idx_non_int','idx':idx,'bad_value':k,'n_premises':n})
            elif k < 1 or k > n:
                issues.append({'dataset':dataset,'record_i':ri,'q_idx':qi,'issue':'idx_out_of_range','idx':idx,'bad_value':k,'n_premises':n})
        if len(idx) != len(set([x for x in idx if isinstance(x,int)])):
            issues.append({'dataset':dataset,'record_i':ri,'q_idx':qi,'issue':'idx_duplicate','idx':idx,'n_premises':n})
    return issues

def fol_stats(rec):
    fols = rec.get('premises-FOL', []) or []
    txt = ' '.join(map(str, fols))
    return {
        'n_fol': len(fols),
        'has_unicode_forall': '∀' in txt,
        'has_unicode_exists': '∃' in txt,
        'has_forall_text': 'ForAll' in txt,
        'has_exists_text': 'Exists' in txt,
        'has_arithmetic': bool(re.search(r'[<>]=?|\b=\b|≥|≤', txt)),
        'has_nested_forall_text': 'ForAll(' in txt and txt.count('ForAll') > 1,
        'fol_empty': len(fols)==0,
    }

def structural_audit(dataset, name):
    issues=[]
    fol_rows=[]
    for ri, rec in enumerate(dataset):
        issues.extend(check_idx(rec, ri, name))
        fs = fol_stats(rec)
        fs.update({'dataset':name,'record_i':ri,'n_premises_NL':len(rec.get('premises-NL', []))})
        if fs['n_fol'] != fs['n_premises_NL']:
            issues.append({'dataset':name,'record_i':ri,'q_idx':None,'issue':'premises_fol_nl_count_mismatch','n_fol':fs['n_fol'],'n_premises_NL':fs['n_premises_NL']})
        fol_rows.append(fs)
    return pd.DataFrame(issues), pd.DataFrame(fol_rows)

raw_struct, raw_fol_stats = structural_audit(raw, 'raw')
clean_struct, clean_fol_stats = structural_audit(clean, 'clean')
struct = pd.concat([raw_struct, clean_struct], ignore_index=True)
folstat = pd.concat([raw_fol_stats, clean_fol_stats], ignore_index=True)

print('='*80)
print('IDX / FOL STRUCTURAL AUDIT')
print('='*80)
print('Structural issues total:', len(struct))
if len(struct):
    display(struct.groupby(['dataset','issue']).size().reset_index(name='count').sort_values(['dataset','count'], ascending=[True,False]))
    display(struct.head(50))
else:
    print('No structural idx/FOL count issues found.')

print('
FOL syntax style counts:')
display(folstat.groupby('dataset')[['has_unicode_forall','has_unicode_exists','has_forall_text','has_exists_text','has_arithmetic','fol_empty']].sum())

struct.to_csv(OUT_DIR/'audit_idx_fol_structural_issues.csv', index=False)
folstat.to_csv(OUT_DIR/'audit_fol_style_stats.csv', index=False)
print('Saved:', OUT_DIR/'audit_idx_fol_structural_issues.csv')
print('Saved:', OUT_DIR/'audit_fol_style_stats.csv')

In [ ]:

# ==================================================================
# STAGE 6 -- Duplicate audit
# ==================================================================
def record_signature(rec, include_answers=False):
    prem = '||'.join(norm_text(x) for x in rec.get('premises-NL', []))
    qs = '||'.join(norm_text(x) for x in rec.get('questions', []))
    ans = '||'.join(norm_answer(x) for x in rec.get('answers', [])) if include_answers else ''
    return h(prem + '###' + qs + '###' + ans, 24)

def duplicate_groups(dataset, name):
    buckets=defaultdict(list)
    for i, rec in enumerate(dataset):
        buckets[record_signature(rec, include_answers=False)].append(i)
    rows=[]
    for sig, inds in buckets.items():
        if len(inds)>1:
            for i in inds:
                rec = dataset[i]
                rows.append({
                    'dataset': name,
                    'signature': sig,
                    'group_size': len(inds),
                    'record_i': i,
                    'answers': rec.get('answers'),
                    'q0': norm_ws(rec.get('questions', [''])[0])[:220] if rec.get('questions') else '',
                    'prem0': norm_ws(rec.get('premises-NL', [''])[0])[:160] if rec.get('premises-NL') else '',
                })
    return pd.DataFrame(rows)

dup_raw = duplicate_groups(raw, 'raw')
dup_clean = duplicate_groups(clean, 'clean')
dups = pd.concat([dup_raw, dup_clean], ignore_index=True)

print('='*80)
print('EXACT DUPLICATE GROUPS BY PREMISES+QUESTIONS')
print('='*80)
print('Raw duplicate rows   :', len(dup_raw), 'groups:', dup_raw.signature.nunique() if len(dup_raw) else 0)
print('Clean duplicate rows :', len(dup_clean), 'groups:', dup_clean.signature.nunique() if len(dup_clean) else 0)
if len(dups):
    display(dups.head(80))

dups.to_csv(OUT_DIR/'audit_duplicate_groups.csv', index=False)
print('Saved:', OUT_DIR/'audit_duplicate_groups.csv')

In [ ]:

# ==================================================================
# STAGE 7 -- High-confidence repair candidates from triangle agreement
# label ↔ explanation ↔ clean changes
# ==================================================================
rf = raw_flags[['record_i','q_idx','answer','explanation_hint','hint_confidence','hint_reason','flag_disagree']].rename(columns={
    'record_i':'raw_record_i','q_idx':'raw_q_idx','answer':'raw_answer_flag','explanation_hint':'raw_exp_hint','hint_confidence':'raw_hint_conf','hint_reason':'raw_hint_reason','flag_disagree':'raw_flag_disagree'
})
cf = clean_flags[['record_i','q_idx','answer','explanation_hint','hint_confidence','hint_reason','flag_disagree']].rename(columns={
    'record_i':'clean_record_i','q_idx':'clean_q_idx','answer':'clean_answer_flag','explanation_hint':'clean_exp_hint','hint_confidence':'clean_hint_conf','hint_reason':'clean_hint_reason','flag_disagree':'clean_flag_disagree'
})
tri = matched.merge(rf, on=['raw_record_i','raw_q_idx'], how='left').merge(cf, on=['clean_record_i','clean_q_idx'], how='left')

def decide_reason(r):
    reasons=[]
    if r.get('answer_changed'):
        reasons.append(f"label_changed:{r.get('raw_answer_norm')}→{r.get('clean_answer_norm')}")
    if bool(r.get('raw_flag_disagree')):
        reasons.append(f"raw_label_vs_exp_hint:{r.get('raw_answer_norm')}!={r.get('raw_exp_hint')}")
    if bool(r.get('clean_flag_disagree')):
        reasons.append(f"clean_label_vs_exp_hint:{r.get('clean_answer_norm')}!={r.get('clean_exp_hint')}")
    if r.get('raw_exp_hint') and r.get('clean_answer_norm') == r.get('raw_exp_hint') and r.get('raw_answer_norm') != r.get('raw_exp_hint'):
        reasons.append('HIGH_CONF_CLEAN_MATCHES_RAW_EXPLANATION_HINT')
    return '; '.join(reasons)

tri['review_reason'] = tri.apply(decide_reason, axis=1)
tri_review = tri[tri['review_reason'].astype(bool)].copy()
tri_review['question_short'] = tri_review['raw_question'].fillna(tri_review['clean_question']).fillna('').map(lambda s: norm_ws(s)[:260])
tri_review['raw_exp_short'] = tri_review['raw_explanation'].fillna('').map(lambda s: norm_ws(s)[:350])
tri_review['clean_exp_short'] = tri_review['clean_explanation'].fillna('').map(lambda s: norm_ws(s)[:350])

print('='*80)
print('TRIANGLE REVIEW CANDIDATES')
print('='*80)
print('Review candidates:', len(tri_review))
print('
Reason counts:')
reason_counter = Counter()
for rs in tri_review['review_reason']:
    for x in str(rs).split('; '):
        if x:
            reason_counter[x.split(':')[0]] += 1
print(reason_counter)

display(tri_review[['raw_record_i','raw_q_idx','clean_record_i','clean_q_idx','raw_answer_norm','clean_answer_norm','raw_exp_hint','clean_exp_hint','review_reason','question_short','raw_exp_short','clean_exp_short']].head(60))

tri_review.to_csv(OUT_DIR/'audit_triangle_review_candidates.csv', index=False)
print('Saved:', OUT_DIR/'audit_triangle_review_candidates.csv')

In [ ]:

# ==================================================================
# STAGE 8 -- Optional Z3-verifier TODO list, not a truth judge
# ==================================================================
print('='*80)
print('Z3 AUDIT POLICY')
print('='*80)
print('''
This notebook intentionally does NOT use Z3 as an absolute judge.
Recommended next notebook/stage:

1. Use only cases with:
   - premises-FOL parse OK
   - question/option formalization grounded 100%
   - no "strongest/fewest premises" comparative wording unless separately modeled
   - idx valid and available

2. Compare three things:
   - label
   - explanation-derived answer hint
   - Z3 entailment result

3. Only mark high-confidence label bugs when at least two independent sources agree, e.g.:
   explanation says A + Z3 proves A, but label is C.

4. Do not auto-fix cases where:
   Z3=Unknown but label=No, unless explanation clearly says insufficient information.
''')

def is_comparative_question(q):
    ql = norm_text(q)
    bad = ['fewest premises', 'strongest conclusion', 'best conclusion', 'most appropriate', 'which conclusion is strongest']
    return any(x in ql for x in bad)

z3_candidates=[]
for _, r in clean_q.iterrows():
    fols = r['premises_FOL'] or []
    fol_text = ' '.join(map(str, fols))
    z3_candidates.append({
        'record_i': r['record_i'],
        'q_idx': r['q_idx'],
        'answer': r['answer_norm'],
        'q_type': 'mc' if r['answer_norm'] in list('ABCD') else 'ynu',
        'comparative_question': is_comparative_question(r['question']),
        'idx_valid_rough': isinstance(r['idx'], list) and all(isinstance(x,int) for x in r['idx']),
        'has_arithmetic_fol': bool(re.search(r'[<>]=?|≥|≤|\b=\b', fol_text)),
        'has_forall_or_exists': any(sym in fol_text for sym in ['∀','∃','ForAll','Exists']),
        'question': norm_ws(r['question'])[:300],
    })
z3_df = pd.DataFrame(z3_candidates)
print('Clean Z3 candidate rough summary:')
display(z3_df.groupby(['q_type','comparative_question','has_arithmetic_fol']).size().reset_index(name='count'))
z3_df.to_csv(OUT_DIR/'audit_z3_candidate_list_clean.csv', index=False)
print('Saved:', OUT_DIR/'audit_z3_candidate_list_clean.csv')

In [ ]:

# ==================================================================
# STAGE 9 -- Final summary and saved artifact list
# ==================================================================
summary.update({
    'matched_questions': int((matched['_merge']=='both').sum()),
    'raw_only_questions': int((matched['_merge']=='left_only').sum()),
    'clean_only_questions': int((matched['_merge']=='right_only').sum()),
    'answer_changed_questions': int(matched['answer_changed'].sum()),
    'raw_expl_label_flags': int(raw_flags.flag_disagree.sum()),
    'clean_expl_label_flags': int(clean_flags.flag_disagree.sum()),
    'structural_issue_count': int(len(struct)),
    'raw_duplicate_groups': int(dup_raw.signature.nunique()) if len(dup_raw) else 0,
    'clean_duplicate_groups': int(dup_clean.signature.nunique()) if len(dup_clean) else 0,
    'triangle_review_candidates': int(len(tri_review)),
})

with open(OUT_DIR/'audit_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print('='*80)
print('FINAL AUDIT SUMMARY')
print('='*80)
for k,v in summary.items():
    if not isinstance(v, dict):
        print(f'{k:32s}: {v}')
print('
Saved files:')
for p in sorted(OUT_DIR.glob('audit_*')):
    print(' -', p)

print('
Recommended reading order:')
print('1. audit_summary.json')
print('2. audit_label_changes_raw_vs_clean.csv')
print('3. audit_triangle_review_candidates.csv')
print('4. audit_idx_fol_structural_issues.csv')
print('5. audit_z3_candidate_list_clean.csv')